In [ ]:
import pandas as pd
import subprocess
import os

In [ ]:
# Release of interest
rel = 12

In [ ]:
wd = '/YOUR/WORKING/DIR'

### Split Female into Regions

In [ ]:
# Release-wide imputed genotypes -- males are already region-stratified
label = 'AJ'
region = 'non_PAR'

males = f'{wd}/releases/release{rel}/imputed_genotypes/{label}/chrX_male_{region}_{label}_release{rel}_vwb'
females = f'{wd}/releases/release{rel}/imputed_genotypes/{label}/chrX_female_{label}_release{rel}_vwb'

In [ ]:
# Split females into region
if region == 'non_PAR':
    chrom = 'X'
elif region == 'PAR':
    chrom = 'PAR1, PAR2'

female_out_path = f'/dev/shm/{label}_females_{region}'

os.chdir('/dev/shm')
! plink2 --pfile {females} --chr {chrom} --make-pgen --out {female_out_path}

In [ ]:
# Check chromosome coding
! tail -n 2 {female_out_path}.pvar
! tail -n 2 {males}.pvar

In [ ]:
# Copy female region files over to Bucket storage
! mkdir -p {wd}/working/R{rel}_prep/XWAS/female/{region}
! scp /dev/shm/{label}_females_{region}.* {wd}/working/R{rel}_prep/XWAS/female/{region}/

### Match Variants

In [ ]:
# Find intersection of variants

os.chdir(wd)

for region in ["non_PAR", "PAR"]:
    male = f'{wd}/releases/release{rel}/imputed_genotypes/{label}/chrX_male_{region}_{label}_release{rel}_vwb'
    female = f'/dev/shm/{label}_females_{region}'

    female_ids = f"female_{region}.ids"
    male_ids   = f"male_{region}.ids"
    common_ids = f"common_{region}.ids"

    # Keep only IDs starting with "chr"
    subprocess.run(
        f"awk 'NR>1{{print $3}}' {female}.pvar | grep -E '^chr' | LC_ALL=C sort -u > {female_ids}",
        shell=True, check=True
    )
    subprocess.run(
        f"awk 'NR>1{{print $3}}' {male}.pvar   | grep -E '^chr' | LC_ALL=C sort -u > {male_ids}",
        shell=True, check=True
    )
    subprocess.run(
        f"comm -12 {female_ids} {male_ids} > {common_ids}",
        shell=True, check=True
    )

In [ ]:
# Check how many variants were kept per sex and region
for region in ['non_PAR', 'PAR']:
    ! wc -l {wd}/female_{region}.ids {wd}/male_{region}.ids {wd}/common_{region}.ids

In [ ]:
# Subset both sexes down to the shared variants
os.chdir('/dev/shm')

for region in ['non_PAR', 'PAR']:
    male = f'{wd}/releases/release{rel}/imputed_genotypes/{label}/chrX_male_{region}_{label}_release{rel}_vwb'
    female = f'{wd}/working/R{rel}_prep/XWAS/female/{region}/{label}_females_{region}'

    ! plink2 --pfile {female} \
        --extract {wd}/common_{region}.ids \
        --make-pgen \
        --sort-vars \
        --out {wd}/working/R{rel}_prep/XWAS/female/{region}/{label}_females_{region}_common_vars

    ! plink2 --pfile {male} \
        --extract {wd}/common_{region}.ids \
        --make-pgen \
        --sort-vars \
        --out {wd}/working/R{rel}_prep/XWAS/male/{region}/{label}_males_{region}_common_vars

In [ ]:
# Combined-sex files carry no 'chr' prefix -- strip it from the shared variant IDs
region = 'PAR'

! sed 's/^chr//' {wd}/common_{region}.ids > {wd}/common_{region}_combined_sex.ids

In [ ]:
# Rebuild variant IDs on the merged combined-sex file so they match the ID list above
combined = f'{wd}/working/R{rel}_prep/XWAS/combined/{region}/{label}_combined_merged_chrX'

! plink2 --pfile {combined} \
  --set-all-var-ids '@:#:$r:$a' \
  --new-id-max-allele-len 500 \
  --make-pgen --out {combined}_FIXED

In [ ]:
! plink2 --pfile {combined}_FIXED \
    --extract {wd}/common_{region}_combined_sex.ids \
    --make-pgen \
    --sort-vars \
    --out {combined}_common_vars

In [ ]:
! tail {combined}_FIXED.pvar

### Sex-Stratified Region-Stratified XWAS

In [ ]:
region = 'PAR'
sex = 'female'

genetic_input = f'{wd}/working/R{rel}_prep/XWAS/{sex}/{region}/{label}_{sex}s_{region}_common_vars'

In [ ]:
# Run command
"""
GLM Modes:
    - xchr-model 1. Male dosages are on a 0..1 scale on chrX, while females are 0..2. This is PLINK 1.9 default.
    - xchr-model 2. Males and females are both on a 0..2 scale on chrX. This is the PLINK 2 default.
PC selection based on Scree Plot in 02_gwas_prep.ipynb
"""

os.chdir('/dev/shm')
gwas_cmd = f'plink2 --pfile {genetic_input} \
--glm hide-covar firth-fallback no-x-sex cols=+a1freq,+a1count \
--xchr-model 2 \
--pheno-name PHENO \
--pheno {wd}/working/R{rel}_prep/XWAS/prep_files/{sex}_{label}_pheno.txt \
--covar {wd}/working/R{rel}_prep/XWAS/prep_files/{sex}_{label}_covar.txt \
--covar-name PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14 \
--covar-variance-standardize \
--out {wd}/working/R{rel}_prep/XWAS/{sex}/{region}/XWAS_{label}_{region}_{sex}'

# Launch in /dev/shm
! {gwas_cmd}

In [ ]:
# Check summary statistics
! head {wd}/working/R{rel}_prep/XWAS/{sex}/{region}/XWAS_{label}_{region}_{sex}.PHENO.glm.logistic.hybrid

### Combined Region-Stratified XWAS

In [ ]:
# Sex-stratified files with matched variants, ready to be merged
label = 'AJ'
region = 'PAR'

final_male = f'{wd}/working/R{rel}_prep/XWAS/male/{region}/{label}_males_{region}_common_vars'
final_female = f'{wd}/working/R{rel}_prep/XWAS/female/{region}/{label}_females_{region}_common_vars'

In [ ]:
# Recode chromosomes numerically so both sexes agree before export
os.chdir('/dev/shm')
out_male = f'/dev/shm/{label}_{region}_males_normalized_common_vars_fixed'
out_female = f'/dev/shm/{label}_{region}_females_normalized_common_vars_fixed'

! plink2 --pfile {final_male} --output-chr 26 --make-pgen --out {out_male}
! plink2 --pfile {final_female} --output-chr 26 --make-pgen --out {out_female}

! tail {out_female}.pvar

In [ ]:
# Export to VCF for merging
male = f'/dev/shm/{label}_{region}_male'
female = f'/dev/shm/{label}_{region}_female'

! plink2 --pfile {out_male} --export vcf 'bgz' --out {male}
! plink2 --pfile {out_female} --export vcf 'bgz' --out {female}

In [ ]:
# Rename the numeric/XY chromosome codes back to chrX
with open('/dev/shm/chr_name_conv.txt', 'w') as f:
    f.write("25 chrX\n")
    f.write("XY chrX\n")

! bcftools annotate --rename-chrs /dev/shm/chr_name_conv.txt {male}.vcf.gz -Oz -o {male}_renamed.vcf.gz
! tabix -p vcf {male}_renamed.vcf.gz

! bcftools annotate --rename-chrs /dev/shm/chr_name_conv.txt {female}.vcf.gz -Oz -o {female}_renamed.vcf.gz
! tabix -p vcf {female}_renamed.vcf.gz

In [ ]:
# Normalize both sexes against the reference before merging
final_male = f'{male}_renamed.vcf.gz'
final_female = f'{female}_renamed.vcf.gz'
fasta_ref = f'{wd}/ref_files/wrayner/hg38.fa'

! htsfile {final_male}
! htsfile {final_female}
! htsfile {fasta_ref}

! bcftools norm --check-ref ws --threads 32 --fasta-ref {fasta_ref} {final_male} -o /dev/shm/{label}_{region}_male_normalized.vcf.gz
! bcftools norm --check-ref ws --threads 32 --fasta-ref {fasta_ref} {final_female} -o /dev/shm/{label}_{region}_female_normalized.vcf.gz

In [ ]:
# Merge the sexes
male = f"/dev/shm/{label}_{region}_male_normalized.vcf.gz"
female = f"/dev/shm/{label}_{region}_female_normalized.vcf.gz"

! tabix -p vcf {male}
! tabix -p vcf {female}
! bcftools merge --threads 32 {female} {male} -o /dev/shm/{label}_merged_chrX_{region}.vcf.gz

In [ ]:
# Split multi-allelics introduced by the merge and copy to Bucket storage
! bcftools norm --check-ref ws --threads 32 --fasta-ref {fasta_ref} -m -any /dev/shm/{label}_merged_chrX_{region}.vcf.gz -o /dev/shm/{label}_merged_chrX_{region}_final.vcf.gz
! tabix -p vcf /dev/shm/{label}_merged_chrX_{region}_final.vcf.gz

! mkdir -p {wd}/working/R{rel}_prep/XWAS/combined/{region}
! scp /dev/shm/{label}_merged_chrX_{region}_final.vcf.gz {wd}/working/R{rel}_prep/XWAS/combined/{region}/

In [ ]:
# The merged VCF carries FID_IID sample names -- rebuild the sex, phenotype and covariate files to match
sex = 'combined'
prep_dir = f'{wd}/working/R{rel}_prep/XWAS/prep_files'

samples = pd.read_csv(f'{wd}/working/R{rel}_prep/imputation/{label}/{label}_release{rel}_{sex}.samples', sep = '\t')
samples['#IID'] = samples['#FID'] + '_' + samples['IID']
samples[['#IID', 'SEX']].to_csv(f'{wd}/working/R{rel}_prep/imputation/{label}/{label}_release{rel}_{sex}_FIXED.samples', index = False, sep = '\t')
display(samples)

pheno = pd.read_csv(f'{prep_dir}/{sex}_{label}_pheno.txt', sep = '\t')
pheno['#IID'] = pheno['#FID'] + '_' + pheno['IID']
pheno[['#IID', 'PHENO']].to_csv(f'{prep_dir}/{sex}_{label}_pheno_FIXED.txt', index = False, sep = '\t')
display(pheno)

covar = pd.read_csv(f'{prep_dir}/{sex}_{label}_covar.txt', sep = '\t')
covar['#IID'] = covar['#FID'] + '_' + covar['IID']
covar.drop(columns = ['#FID', 'IID'], inplace = True)
cols = ['#IID'] + [c for c in covar.columns if c != '#IID']
covar[cols].to_csv(f'{prep_dir}/{sex}_{label}_covar_FIXED.txt', index = False, sep = '\t')
display(covar)

In [ ]:
# Convert the merged VCF back to PLINK
os.chdir(wd)

# Establish paths
merged_path = f'{wd}/working/R{rel}_prep/XWAS/combined/{region}/{label}_merged_chrX_{region}_final.vcf.gz'
sex_values = f'{wd}/working/R{rel}_prep/imputation/{label}/{label}_release{rel}_{sex}_FIXED.samples'
intermed_dir = f'/dev/shm/chrX_{label}_rel{rel}_merged_{region}'
out_dir = f'{wd}/working/R{rel}_prep/XWAS/combined/{region}/'

if region == 'PAR':
    subset_cmd = f"plink2 --vcf {merged_path} 'dosage=HDS' --update-sex {sex_values} --split-par b38 --make-pgen --out {intermed_dir}"
else:
    subset_cmd = f"plink2 --vcf {merged_path} --update-sex {sex_values} --split-par b38 --make-pgen --out {intermed_dir}"
cp_cmd = f"scp {intermed_dir}.* {out_dir}"

with open(f'{wd}/convert_vcf_to_pfile.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write(f'{subset_cmd}\n')
    f.write(f'{cp_cmd}\n')
    f.close()

print(f"Preparing bash script for converting {label} ancestry!")

In [ ]:
# Launch in separate shell windows
! chmod +x {wd}/convert_vcf_to_pfile.sh
! {wd}/convert_vcf_to_pfile.sh

In [ ]:
region = 'non_PAR'
sex = 'combined'

genetics_input = f'{wd}/working/R{rel}_prep/XWAS/combined/{region}/chrX_{label}_rel{rel}_merged_{region}'

! head {genetics_input}.psam

In [ ]:
# Run command -- SEX is added as a covariate when both sexes are analyzed together
"""
GLM Modes:
    - xchr-model 1. Male dosages are on a 0..1 scale on chrX, while females are 0..2. This is PLINK 1.9 default.
    - xchr-model 2. Males and females are both on a 0..2 scale on chrX. This is the PLINK 2 default.
PC selection based on Scree Plot in 02_gwas_prep.ipynb
"""

os.chdir('/dev/shm')
gwas_cmd = f'plink2 --pfile {genetics_input} \
--glm hide-covar firth-fallback no-x-sex cols=+a1freq,+a1count \
--xchr-model 2 \
--pheno-name PHENO \
--pheno {wd}/working/R{rel}_prep/XWAS/prep_files/{sex}_{label}_pheno_FIXED.txt \
--covar {wd}/working/R{rel}_prep/XWAS/prep_files/{sex}_{label}_covar_FIXED.txt \
--covar-name SEX,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9 \
--covar-variance-standardize \
--out {wd}/working/R{rel}_prep/XWAS/combined/{region}/XWAS_{label}_{region}_{sex}'

# Launch in /dev/shm
! {gwas_cmd}